## Consolidación de base_clean.py v1

Junto en un solo lugar todo lo que fui confirmando en el diagnóstico: la
función de carga con el fix de columnas de `devices`, las funciones de
diagnóstico (pandas para tablas chicas, SQL para `transactions` por su
volumen), los chequeos puntuales que usé para investigar cada hallazgo, y
finalmente las funciones `clean_*` por tabla con los filtros ya decididos
y justificados:

- `clean_devices`: corrige el renombre de columnas y elimina la fila de
  header que se coló como dato
- `clean_users`: descarta `num_referrals`/`num_successful_referrals` (cero
  varianza, sin poder predictivo), calcula `age` con `REFERENCE_DATE` fija,
  y aplica el filtro de edad plausible [18, 122] como salvaguarda aunque
  hoy no elimine ninguna fila
- `clean_notifications`: pasa sin cambios, no encontré nada que limpiar
- `clean_transactions`: filtra montos ≤0 y el bug de conversión de moneda
  (>$1M, confinado a `DECLINED`), y agrego `has_merchant_info` para no
  confundir los nulls estructurales de `ea_*` con datos faltantes reales

Dejo pendientes dos decisiones que quiero resolver contigo antes de congelar
esta versión: cómo tratar los nulls de `marketing_push`/`marketing_email`
(¿imputar como "unknown"?) y los nulls de `ea_*` (¿NaN explícito o
"NOT_APPLICABLE"?).

In [3]:
from google.cloud import bigquery
import pandas as pd
from datetime import date

# -----------------------------------------------------------------------
# Configuración general
# -----------------------------------------------------------------------
PROJECT_ID = "quiet-seer-299408"
DATASET = "neobank"

# Fecha de referencia FIJA para todos los cálculos de edad/antigüedad.
REFERENCE_DATE = date(2025, 1, 1)  # TODO: confirmar fecha exacta a usar

client = bigquery.Client(project=PROJECT_ID)


def load_table(table_name: str) -> pd.DataFrame:
    """Carga una tabla completa del dataset neobank como DataFrame."""
    query = f"SELECT * FROM `{PROJECT_ID}.{DATASET}.{table_name}`"
    return client.query(query).to_dataframe()


def load_devices() -> pd.DataFrame:
    """
    Carga `devices` renombrando las columnas mal inferidas por BigQuery.
    string_field_0 -> device_brand (confirmado por muestra: 'Apple', etc.)
    string_field_1 -> user_id
    """
    query = f"""
        SELECT
            string_field_1 AS user_id,
            string_field_0 AS device_brand
        FROM `{PROJECT_ID}.{DATASET}.devices`
    """
    return client.query(query).to_dataframe()


# -----------------------------------------------------------------------
# Diagnóstico de calidad de datos
# -----------------------------------------------------------------------

def diagnose(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "n_nulls": df.isna().sum(),
        "pct_nulls": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(),
    })
    n_dupes = df.duplicated().sum()
    print(f"--- {table_name} ---")
    print(f"Filas: {len(df):,} | Filas duplicadas (100% match): {n_dupes:,}")
    return summary


def diagnose_sql(table_name: str) -> pd.DataFrame:
    table_ref = client.get_table(f"{PROJECT_ID}.{DATASET}.{table_name}")
    columns = [field.name for field in table_ref.schema]
    null_counts_sql = ",\n            ".join(
        f"COUNTIF({col} IS NULL) AS nulls_{col}" for col in columns
    )
    query = f"""
        SELECT
            COUNT(*) AS n_rows,
            COUNT(*) - COUNT(DISTINCT TO_JSON_STRING(t)) AS n_dupes,
            {null_counts_sql}
        FROM `{PROJECT_ID}.{DATASET}.{table_name}` AS t
    """
    result = client.query(query).to_dataframe().iloc[0]
    n_rows = result["n_rows"]
    summary = pd.DataFrame({
        "n_nulls": [result[f"nulls_{col}"] for col in columns],
    }, index=columns)
    summary["pct_nulls"] = (summary["n_nulls"] / n_rows * 100).round(2)
    print(f"--- {table_name} (calculado en SQL) ---")
    print(f"Filas: {n_rows:,} | Filas duplicadas (100% match): {result['n_dupes']:,}")
    return summary


# -----------------------------------------------------------------------
# Diagnósticos dirigidos
# -----------------------------------------------------------------------

def check_constant_columns(df: pd.DataFrame, cols: list[str]) -> None:
    for col in cols:
        print(f"{col}: valor único = {df[col].unique()}")


def check_orphans(child_table: str, parent_table: str = "users") -> pd.DataFrame:
    if child_table == "devices":
        from_clause = f"""(
            SELECT string_field_1 AS user_id
            FROM `{PROJECT_ID}.{DATASET}.devices`
        )"""
    else:
        from_clause = f"`{PROJECT_ID}.{DATASET}.{child_table}`"
    query = f"""
        SELECT DISTINCT c.user_id
        FROM {from_clause} AS c
        LEFT JOIN `{PROJECT_ID}.{DATASET}.{parent_table}` AS p
            ON c.user_id = p.user_id
        WHERE p.user_id IS NULL
    """
    return client.query(query).to_dataframe()


def check_null_overlap(df: pd.DataFrame, col_a: str, col_b: str) -> None:
    null_a = df[col_a].isna()
    null_b = df[col_b].isna()
    print(f"Nulls en {col_a}: {null_a.sum()} | Nulls en {col_b}: {null_b.sum()}")
    print(f"Nulls en ambas a la vez: {(null_a & null_b).sum()}")


def null_pattern_by_group_sql(table_name: str, null_col: str, group_col: str) -> pd.DataFrame:
    query = f"""
        SELECT
            {group_col},
            COUNT(*) AS n_rows,
            COUNTIF({null_col} IS NULL) AS n_nulls,
            ROUND(COUNTIF({null_col} IS NULL) / COUNT(*) * 100, 2) AS pct_nulls
        FROM `{PROJECT_ID}.{DATASET}.{table_name}`
        GROUP BY {group_col}
        ORDER BY n_rows DESC
    """
    return client.query(query).to_dataframe()


def top_values_sql(table_name: str, col: str, n: int = 20) -> pd.DataFrame:
    query = f"""
        SELECT * FROM `{PROJECT_ID}.{DATASET}.{table_name}`
        ORDER BY {col} DESC LIMIT {n}
    """
    return client.query(query).to_dataframe()


def threshold_counts_sql(table_name: str, col: str, thresholds: list[float]) -> pd.DataFrame:
    counts_sql = ",\n            ".join(
        f"COUNTIF({col} > {t}) AS n_above_{int(t) if t == int(t) else t}"
        for t in thresholds
    )
    query = f"""
        SELECT COUNT(*) AS n_total, {counts_sql}
        FROM `{PROJECT_ID}.{DATASET}.{table_name}`
    """
    return client.query(query).to_dataframe()


def range_by_group_sql(table_name: str, col: str, group_col: str) -> pd.DataFrame:
    query = f"""
        SELECT
            {group_col}, COUNT(*) AS n_rows, MIN({col}) AS min_val,
            MAX({col}) AS max_val, AVG({col}) AS avg_val,
            COUNTIF({col} > 50000) AS n_above_50000
        FROM `{PROJECT_ID}.{DATASET}.{table_name}`
        GROUP BY {group_col} ORDER BY max_val DESC
    """
    return client.query(query).to_dataframe()


# -----------------------------------------------------------------------
# Funciones de limpieza por tabla (nombres de salida en inglés)
# -----------------------------------------------------------------------

CARD_RELATED_TYPES = ["CARD_PAYMENT", "ATM", "CARD_REFUND"]
AMOUNT_USD_UPPER_BOUND = 1_000_000
MIN_AGE = 18
MAX_AGE = 122  # récord de longevidad verificado (Jeanne Calment, GRG)


def clean_devices() -> pd.DataFrame:
    """Renombra columnas y elimina la fila de header mal cargada."""
    df = load_devices()
    df = df[df["user_id"] != "user_id"].reset_index(drop=True)
    return df


def clean_users() -> pd.DataFrame:
    """Descarta columnas sin varianza, calcula age, aplica filtro 18-122."""
    df = load_table("users")
    df["age"] = REFERENCE_DATE.year - df["birth_year"]
    df = df[(df["age"] >= MIN_AGE) & (df["age"] <= MAX_AGE)].reset_index(drop=True)
    df = df.drop(columns=["num_referrals", "num_successful_referrals"])
    return df


def clean_notifications() -> pd.DataFrame:
    """Sin hallazgos de calidad que requieran filtro."""
    return load_table("notifications")


def clean_transactions() -> pd.DataFrame:
    """Filtra amount_usd <=0 y >1M (bug de conversión); marca has_merchant_info."""
    df = load_table("transactions")
    df = df[
        (df["amount_usd"] > 0) & (df["amount_usd"] <= AMOUNT_USD_UPPER_BOUND)
    ].reset_index(drop=True)
    df["has_merchant_info"] = df["transactions_type"].isin(CARD_RELATED_TYPES)
    return df

In [4]:
clean_users_df = clean_users()
clean_devices_df = clean_devices()
clean_notifications_df = clean_notifications()
clean_transactions_df = clean_transactions()

print("users:", clean_users_df.shape)
print("devices:", clean_devices_df.shape)
print("notifications:", clean_notifications_df.shape)
print("transactions:", clean_transactions_df.shape)
print("\nTransacciones eliminadas por filtro amount_usd:", 2_740_075 - len(clean_transactions_df))
print("has_merchant_info counts:\n", clean_transactions_df["has_merchant_info"].value_counts())

users: (19430, 11)
devices: (19430, 2)
notifications: (121813, 5)
transactions: (2681355, 13)

Transacciones eliminadas por filtro amount_usd: 58720
has_merchant_info counts:
 has_merchant_info
True     1532062
False    1149293
Name: count, dtype: int64


## Validación del filtro de amount_usd

Las 58,720 transacciones eliminadas coinciden exactamente con lo esperado:
58,604 filas con amount_usd <= 0 (confirmado en el diagnóstico inicial) +
116 filas con amount_usd > $1,000,000 (confirmado como el bug de conversión
confinado a DECLINED) = 58,720. Que el número cuadre exacto con lo que ya
había visto en el diagnóstico me da confianza de que el filtro se aplicó
correctamente y no está eliminando de más ni de menos.

## age_group para curvas de supervivencia y dashboard

Agrego age_group como variable categórica adicional, sin eliminar age
continua. Elegí cortes basados en los cuartiles reales de la distribución
(no rangos redondos tipo "18-30, 31-40") porque así cada grupo queda con
un tamaño de muestra similar -- esto me da más poder estadístico cuando
compare curvas de Kaplan-Meier entre grupos, en vez de tener un grupo con
miles de usuarios y otro con veinte. Guardo age_group solo para las curvas
KM, el Chi-square exploratorio y el dashboard -- para Random Forest, Cox y
PCA sigo usando age continua, porque binear ahí solo tiraría información
que el modelo puede aprovechar mejor por sí mismo.